In [1]:
from pathlib import Path
from dmpbridge.evaluation.pdfplumber_text_evaluator import evaluate_pdfplumber_text

result = evaluate_pdfplumber_text(
    extracted_txt_path=Path("../data/pdfplumber_extracted_text/sample4.txt"),
    reference_txt_path=Path("../data/reference_text/sample4_reference.txt"),
)

result

{'sample_id': 'sample4',
 'word_capture': 0.997,
 'rouge_l': 1.0,
 'extracted_word_count': 348,
 'reference_word_count': 347,
 'missing_word_count': 1,
 'missing_words_preview': 'platformindependent',
 'extracted_line_count': 86,
 'reference_line_count': 78}

In [2]:
import pandas as pd
from pathlib import Path

from dmpbridge.evaluation.pdfplumber_text_evaluator import evaluate_pdfplumber_text

results = []

extracted_folder = Path("../data/pdfplumber_extracted_text")
reference_folder = Path("../data/reference_text")

for extracted_file in sorted(extracted_folder.glob("*.txt")):

    sample_id = extracted_file.stem
    reference_file = reference_folder / f"{sample_id}_reference.txt"

    if not reference_file.exists():
        print(f"Missing reference file: {reference_file}")
        continue

    result = evaluate_pdfplumber_text(
        extracted_txt_path=extracted_file,
        reference_txt_path=reference_file,
    )

    results.append(result)

df = pd.DataFrame(results)

display(df)

,sample_id,word_capture,rouge_l,extracted_word_count,reference_word_count,missing_word_count,missing_words_preview,extracted_line_count,reference_line_count
0,sample1,1.000,0.996,272,272,0,,87,40
1,sample10,0.997,0.999,347,344,1,also,76,68
2,sample2,0.998,1.000,615,615,1,rom,191,71
3,sample3,1.000,1.000,318,317,0,,81,69
4,sample4,0.997,1.000,348,347,1,platformindependent,86,78
5,sample5,0.997,0.997,394,392,1,recordlevel,89,80
6,sample6,0.992,1.000,132,132,1,opensource,28,24
7,sample7,1.000,1.000,132,131,0,,21,17
8,sample8,1.000,1.000,328,327,0,,67,59
9,sample9,0.998,1.000,439,438,1,publiclyavailable,97,85


In [5]:
from pathlib import Path
import re
import time
import difflib
import pandas as pd

extracted_folder = Path("../data/pdfplumber_extracted_text")
reference_folder = Path("../data/reference_text")

output_folder = Path("../outputs/evaluation")
output_folder.mkdir(parents=True, exist_ok=True)

report_path = output_folder / "pdfplumber_paper_style_metrics.csv"


def normalize_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def split_paragraphs(text):
    return [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]


def tokenize_words(text):
    text = text.lower()
    text = re.sub(r"[^\w\s'-]", " ", text)
    return re.findall(r"\b\w+(?:[-']\w+)?\b", text)


def safe_percent(value, denominator):
    return 0.0 if denominator == 0 else round((value / denominator) * 100, 2)


def get_extracted_filename(reference_path):
    # sample1_reference.txt -> sample1.txt
    return reference_path.name.replace("_reference", "")


def word_diff_metrics(reference_words, extracted_words):
    matcher = difflib.SequenceMatcher(None, reference_words, extracted_words)

    w_plus = 0
    w_minus = 0
    w_tilde = 0

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        ref_chunk = reference_words[i1:i2]
        ext_chunk = extracted_words[j1:j2]

        if tag == "insert":
            w_plus += len(ext_chunk)

        elif tag == "delete":
            w_minus += len(ref_chunk)

        elif tag == "replace":
            min_len = min(len(ref_chunk), len(ext_chunk))

            for k in range(min_len):
                similarity = difflib.SequenceMatcher(
                    None, ref_chunk[k], ext_chunk[k]
                ).ratio()

                if similarity >= 0.75:
                    w_tilde += 1
                else:
                    w_minus += 1
                    w_plus += 1

            w_minus += max(0, len(ref_chunk) - min_len)
            w_plus += max(0, len(ext_chunk) - min_len)

    return w_plus, w_minus, w_tilde


def newline_metrics(reference_text, extracted_text):
    reference_newlines = reference_text.count("\n")
    extracted_newlines = extracted_text.count("\n")

    nl_plus = max(0, extracted_newlines - reference_newlines)
    nl_minus = max(0, reference_newlines - extracted_newlines)

    return nl_plus, nl_minus, reference_newlines


def normalize_paragraph_for_match(paragraph):
    return " ".join(tokenize_words(paragraph))


def paragraph_similarity(p1, p2):
    return difflib.SequenceMatcher(
        None,
        normalize_paragraph_for_match(p1),
        normalize_paragraph_for_match(p2),
    ).ratio()


def paragraph_metrics(reference_paragraphs, extracted_paragraphs, similarity_threshold=0.70):
    matched_reference = set()
    matched_extracted = set()
    matches = []

    for ext_i, ext_p in enumerate(extracted_paragraphs):
        best_ref_i = None
        best_score = 0

        for ref_i, ref_p in enumerate(reference_paragraphs):
            if ref_i in matched_reference:
                continue

            score = paragraph_similarity(ref_p, ext_p)

            if score > best_score:
                best_score = score
                best_ref_i = ref_i

        if best_ref_i is not None and best_score >= similarity_threshold:
            matched_reference.add(best_ref_i)
            matched_extracted.add(ext_i)
            matches.append((ext_i, best_ref_i, best_score))

    p_plus = len(extracted_paragraphs) - len(matched_extracted)
    p_minus = len(reference_paragraphs) - len(matched_reference)

    sorted_matches = sorted(matches, key=lambda x: x[0])
    reference_order = [ref_i for _, ref_i, _ in sorted_matches]

    p_reordered = 0
    for i in range(1, len(reference_order)):
        if reference_order[i] < reference_order[i - 1]:
            p_reordered += 1

    return p_plus, p_minus, p_reordered


def compute_z_score(
    nl_plus, nl_minus,
    w_plus, w_minus, w_tilde,
    p_plus, p_minus, p_reordered,
    c=5
):
    return (
        nl_plus + nl_minus
        + w_plus + w_minus + w_tilde
        + c * (p_plus + p_minus + p_reordered)
    )


def evaluate_file(reference_path, extracted_path):
    start_time = time.time()

    reference_text = normalize_text(
        reference_path.read_text(encoding="utf-8", errors="ignore")
    )

    extracted_text = normalize_text(
        extracted_path.read_text(encoding="utf-8", errors="ignore")
    )

    runtime = time.time() - start_time

    reference_words = tokenize_words(reference_text)
    extracted_words = tokenize_words(extracted_text)

    reference_paragraphs = split_paragraphs(reference_text)
    extracted_paragraphs = split_paragraphs(extracted_text)

    gt_words = len(reference_words)

    nl_plus, nl_minus, gt_newlines = newline_metrics(reference_text, extracted_text)
    p_plus, p_minus, p_reordered = paragraph_metrics(reference_paragraphs, extracted_paragraphs)
    w_plus, w_minus, w_tilde = word_diff_metrics(reference_words, extracted_words)

    z_score = compute_z_score(
        nl_plus, nl_minus,
        w_plus, w_minus, w_tilde,
        p_plus, p_minus, p_reordered,
        c=5
    )

    return {
        "file_name": reference_path.stem,
        "reference_file": reference_path.name,
        "extracted_file": extracted_path.name,

        "GT_words": gt_words,
        "GT_newlines": gt_newlines,
        "GT_paragraphs": len(reference_paragraphs),

        "Extracted_words": len(extracted_words),
        "Extracted_paragraphs": len(extracted_paragraphs),

        "NL+": nl_plus,
        "NL+_%": safe_percent(nl_plus, gt_newlines),
        "NL-": nl_minus,
        "NL-_%": safe_percent(nl_minus, gt_newlines),

        "P+": p_plus,
        "P+_%": safe_percent(p_plus, gt_words),
        "P-": p_minus,
        "P-_%": safe_percent(p_minus, gt_words),
        "P_reordered": p_reordered,
        "P_reordered_%": safe_percent(p_reordered, gt_words),

        "W+": w_plus,
        "W+_%": safe_percent(w_plus, gt_words),
        "W-": w_minus,
        "W-_%": safe_percent(w_minus, gt_words),
        "W_misspelled": w_tilde,
        "W_misspelled_%": safe_percent(w_tilde, gt_words),

        "ERR": 0,
        "T_seconds": round(runtime, 4),
        "Z_score_c5": z_score,
    }


results = []

reference_files = sorted(reference_folder.glob("*.txt"))

for reference_path in reference_files:
    extracted_filename = get_extracted_filename(reference_path)
    extracted_path = extracted_folder / extracted_filename

    print("Reference:", reference_path.name, "-> Extracted:", extracted_filename)

    if not extracted_path.exists():
        results.append({
            "file_name": reference_path.stem,
            "reference_file": reference_path.name,
            "expected_extracted_file": extracted_filename,
            "ERR": 1,
            "error_message": "Missing extracted pdfplumber text file"
        })
        continue

    try:
        results.append(evaluate_file(reference_path, extracted_path))

    except Exception as e:
        results.append({
            "file_name": reference_path.stem,
            "reference_file": reference_path.name,
            "expected_extracted_file": extracted_filename,
            "ERR": 1,
            "error_message": str(e)
        })

df_metrics = pd.DataFrame(results)
df_metrics.to_csv(report_path, index=False)

print("Evaluation completed.")
print("Reference files:", len(reference_files))
print("Saved report to:", report_path)

df_metrics

Reference: sample10_reference.txt -> Extracted: sample10.txt
Reference: sample1_reference.txt -> Extracted: sample1.txt
Reference: sample2_reference.txt -> Extracted: sample2.txt
Reference: sample3_reference.txt -> Extracted: sample3.txt
Reference: sample4_reference.txt -> Extracted: sample4.txt
Reference: sample5_reference.txt -> Extracted: sample5.txt
Reference: sample6_reference.txt -> Extracted: sample6.txt
Reference: sample7_reference.txt -> Extracted: sample7.txt
Reference: sample8_reference.txt -> Extracted: sample8.txt
Reference: sample9_reference.txt -> Extracted: sample9.txt
Evaluation completed.
Reference files: 10
Saved report to: ..\outputs\evaluation\pdfplumber_paper_style_metrics.csv


,file_name,reference_file,extracted_file,GT_words,GT_newlines,GT_paragraphs,Extracted_words,Extracted_paragraphs,NL+,NL+_%,...,P_reordered_%,W+,W+_%,W-,W-_%,W_misspelled,W_misspelled_%,ERR,T_seconds,Z_score_c5
0,sample10_reference,sample10_reference.txt,sample10.txt,890,67,1,891,4,5,7.46,...,0.0,7,0.79,6,0.67,0,0.0,0,0.000,33
1,sample1_reference,sample1_reference.txt,sample1.txt,991,39,1,997,4,44,112.82,...,0.0,8,0.81,2,0.20,0,0.0,0,0.000,79
2,sample2_reference,sample2_reference.txt,sample2.txt,2165,59,16,2174,10,125,211.86,...,0.0,11,0.51,2,0.09,0,0.0,0,0.001,248
3,sample3_reference,sample3_reference.txt,sample3.txt,848,68,1,854,6,8,11.76,...,0.0,6,0.71,0,0.00,0,0.0,0,0.001,49
4,sample4_reference,sample4_reference.txt,sample4.txt,1015,77,1,1020,4,5,6.49,...,0.0,6,0.59,1,0.10,0,0.0,0,0.000,37
5,sample5_reference,sample5_reference.txt,sample5.txt,1072,79,1,1077,4,6,7.59,...,0.0,6,0.56,1,0.09,0,0.0,0,0.001,28
6,sample6_reference,sample6_reference.txt,sample6.txt,284,23,1,287,2,2,8.70,...,0.0,4,1.41,1,0.35,0,0.0,0,0.000,12
7,sample7_reference,sample7_reference.txt,sample7.txt,259,16,1,261,2,2,12.50,...,0.0,2,0.77,0,0.00,0,0.0,0,0.001,9
8,sample8_reference,sample8_reference.txt,sample8.txt,835,58,1,839,4,5,8.62,...,0.0,4,0.48,0,0.00,0,0.0,0,0.000,24
9,sample9_reference,sample9_reference.txt,sample9.txt,1105,84,1,1112,6,8,9.52,...,0.0,8,0.72,1,0.09,0,0.0,0,0.000,52
